# Drex hands-on

Drex (NaceAI) answers the same way as Jev: facts in, a probability for every option out. Nace documents it as wire-compatible with TypeSafe.

From the setup prompt on [nace.ai/drex](https://www.nace.ai/drex):

- Base URL: `https://drex.nace.ai`
- `POST /v1/systemone`
- `Authorization: Bearer $DREX_API_KEY`
- Model: `drex-latest`

Question types are the same three: `choice`, `score`, `noul`.

Sign in at [drex.nace.ai](https://drex.nace.ai)  Same exercises on Jev: [jev_hands_on.ipynb](jev_hands_on.ipynb).

In [1]:
import os

os.environ["DREX_API_KEY"] = "nace_sk_***"  

API_KEY = os.environ["DREX_API_KEY"].strip()
ENDPOINT = "https://drex.nace.ai/v1/systemone"
MODEL = "drex-latest"

assert API_KEY, "Set DREX_API_KEY in this cell"

## One call

Same body as Jev: `state`, `model`, `questions`. This ticket is the one Nace recorded on their Drex page.

In [4]:
import json
import urllib.request


def decide(state, questions):
    body = json.dumps({"model": MODEL, "state": state, "questions": questions}).encode()
    req = urllib.request.Request(
        ENDPOINT,
        data=body,
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        return json.load(resp)


TICKET = (
    "Customer: I was charged twice last Tuesday and I am furious. "
    "Order #4419. I want a refund today."
)

QUESTIONS = {
    "topic": {
        "type": "choice",
        "instructions": "What is the issue about?",
        "criteria": {
            "billing": "Charges, refunds, invoices",
            "bug": "Something in the product is broken",
            "shipping": "Delivery or a missing order",
        },
    },
    "frustration": {
        "type": "score",
        "instructions": "How frustrated does the customer sound?",
        "criteria": ["Calm", "Annoyed", "Furious"],
    },
    "escalate": {
        "type": "noul",
        "instructions": "Escalate to a human now?",
    },
}

result = decide(TICKET, QUESTIONS)
print("model:", result["model"])
print("tokens:", result.get("usage"))
print(json.dumps(result["answers"], indent=2))

model: drex-v1.0
tokens: {'input_tokens': 101, 'output_tokens': 181}
{
  "topic": {
    "type": "choice",
    "choice": "billing",
    "confidence": 0.9972,
    "probabilities": {
      "billing": 0.9981,
      "bug": 0.0004,
      "shipping": 0.0014
    }
  },
  "frustration": {
    "type": "score",
    "score": 1.9571,
    "legend": [
      "Calm",
      "Annoyed",
      "Furious"
    ],
    "probabilities": {
      "0": 0.0009,
      "1": 0.0412,
      "2": 0.9579
    },
    "confidence": 0.9356
  },
  "escalate": {
    "type": "noul",
    "noul": 0.8395
  }
}


## Act on the probabilities

Same rule as the Jev notebook, so you can compare the two numbers on one ticket. `0.8` is your cutoff, not the model's.

In [6]:
answers = result["answers"]
topic = answers["topic"]
mood = answers["frustration"]
escalate_p = answers["escalate"]["noul"]

print("topic:", topic["choice"], topic["probabilities"])
print("frustration score:", mood["score"], "legend:", mood["legend"])
print("escalate probability:", escalate_p)

if escalate_p >= 0.8:
    action = "send to a human"
else:
    action = f"route to {topic['choice']}"
print("action:", action)

topic: billing {'billing': 0.9981, 'bug': 0.0004, 'shipping': 0.0014}
frustration score: 1.9571 legend: ['Calm', 'Annoyed', 'Furious']
escalate probability: 0.8395
action: send to a human


## Your turn

Run the same text in the Jev notebook. Compare `choice`, `noul`, and `usage.input_tokens`.

In [7]:
mine = decide(
    "The export button crashes Safari. No workaround. Customers cannot download invoices.",
    {
        "team": {
            "type": "choice",
            "instructions": "Which team should take this?",
            "criteria": {
                "billing": "Money movement",
                "engineering": "Product bug",
                "support": "How-to, no code change",
            },
        },
        "blocking": {
            "type": "noul",
            "instructions": "Is a core workflow blocked?",
        },
    },
)
print(json.dumps(mine["answers"], indent=2))

{
  "team": {
    "type": "choice",
    "choice": "engineering",
    "confidence": 0.9513,
    "probabilities": {
      "billing": 0.0148,
      "engineering": 0.9675,
      "support": 0.0176
    }
  },
  "blocking": {
    "type": "noul",
    "noul": 0.9606
  }
}
